# Mikołaj: Causal Tracing na gpt2-xl (Część I, pkt 1–2)

Zakres (3 pkt), szczegóły w [mikolaj.md](mikolaj.md) i [instrukcji](../Zadanie_4.md):

1. **I.1–3 [1.5 pkt]**: notatki z surveyu KE (definicja zadania, metryki), ROME / Causal Tracing, pyvene 101 (+ notebook #1).
2. **Plan pkt 1**: odtworzenie notebooka #2 (`Causal_Tracing.ipynb`): Space Needle → Seattle na gpt2-xl.
3. **Plan pkt 2 [1.5 pkt]**: 5 własnych faktów, zaszumiony **tylko podmiot**, wykresy `block_output` / `mlp_activation` / `attention_output`.
4. Odpowiedzi na pytania pomocnicze.

**Zasada:** cały kod jest w `../utils/tracing/` (pętla, szum, analiza, wykresy). Tu tylko wywołania i wnioski.
Wyniki (CSV/JSON) trafiają do `artifacts/mikolaj/` i są cache'owane. **Dominik czyta je do porównania**, więc nie zmieniaj nazw (`name=`) po zapisaniu. Końcowe wykresy: `figures/*.png`.

In [ ]:
import sys

sys.path.insert(0, "..")

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_formats = ['retina']

In [ ]:
import pandas as pd

from utils.infra import artifacts_dir, get_device, load_gpt2_xl, set_seed
from utils.tracing import (
    NOTEBOOK_NOISE_LEVEL_GPT2_XL,
    compute_noise_level,
    find_token_positions,
    next_token_prob,
    position_summary,
    run_corrupted,
    streams_summary,
    tokenize,
    top_next_tokens,
    trace_all_streams,
)
from utils.tracing.plot import plot_trace_grid, plot_trace_streams

set_seed(42)
OUT = artifacts_dir("mikolaj")
FIGURES = "figures"
device = get_device()
device

## I.1: survey *Knowledge Editing for LLMs* (definicja zadania, metryki)

TODO: definicja edycji wiedzy (krotka *(s, r, o) → (s, r, o\*)*, zakres edycji), metryki: accuracy, locality, generality, portability, retainability, scalability.

## I.2: ROME i Causal Tracing

TODO: trzy przebiegi (clean / corrupted / corrupted-with-restoration), efekt pośredni (IE), wnioski z artykułu (MLP w środkowych warstwach na ostatnim tokenie podmiotu, atencja na ostatnim tokenie), jak ROME edytuje MLP.

## I.3: pyvene 101 i notebook #1 (`Basic_Intervention.ipynb`)

TODO: `IntervenableConfig`, `RepresentationConfig`, `IntervenableModel`, `unit_locations`, typy interwencji. Jak to się przekłada na `utils/tracing/interventions.py`.

## Plan pkt 1: odtworzenie notebooka #2 (Space Needle)

gpt2-xl ma ok. 6,5 GB; pierwsze pobranie trwa kilka minut.

In [ ]:
model, tok = load_gpt2_xl(device)
print(model.config.n_layer, "warstw,", model.config.n_embd, "wymiarów")

In [ ]:
text = "The Space Needle is in downtown"
subject = find_token_positions(tok, text, "The Space Needle")
print(tokenize(tok, text), "-> podmiot:", subject)
top_next_tokens(model, tok, text, k=10)

Poziom szumu: stała z notebooka #2 (3 × std osadzeń podmiotów, policzona przez autorów) vs 3 × std całej macierzy osadzeń. Do reprodukcji używamy stałej.

In [ ]:
NOISE = NOTEBOOK_NOISE_LEVEL_GPT2_XL
print(f"notebook: {NOISE:.4f}, 3*std(wte): {compute_noise_level(model):.4f}")

seattle = next_token_prob(tok, " Seattle")
print("P(Seattle) po zaszumieniu:", run_corrupted(model, tok, text, subject, seattle, NOISE))

In [ ]:
space = trace_all_streams(
    model, tok, text, subject, seattle,
    out_dir=OUT, name="space_needle", noise_level=NOISE,
)
plot_trace_streams(space, save_path=f"{FIGURES}/space_needle.png");

In [ ]:
streams_summary(space)

TODO: porównanie z Rys. 1 w ROME (czy widać oba obszary: środkowe MLP na ostatnim tokenie podmiotu i późne warstwy na ostatnim tokenie).

## Plan pkt 2: 5 własnych faktów

Warunki: fakt jest faktograficzny, odpowiedź to **pierwszy token** po prompcie (ze spacją), model zna fakt (odpowiedź w top-3), zaszumiamy **tylko podmiot**.
Propozycje w [mikolaj.md](mikolaj.md#propozycje-faktów-do-wyboru--zmiany): wybierz 5 i zmień według uznania.

In [ ]:
# (nazwa, prompt, podmiot, odpowiedź). TODO: wybrać / zmienić
FACTS = [
    ("eiffel", "The Eiffel Tower is located in the city of", "The Eiffel Tower", " Paris"),
    ("obama", "Barack Obama was born in the state of", "Barack Obama", " Hawaii"),
    ("brazil", "The official language of Brazil is", "Brazil", " Portuguese"),
    ("jobs", "Steve Jobs was the co-founder of", "Steve Jobs", " Apple"),
    ("messi", "Lionel Messi plays the sport of", "Lionel Messi", " football"),
]

Sprawdzenie, czy model zna fakty (bez tego wykres nic nie pokaże):

In [ ]:
rows = []
for name, prompt, subj, answer in FACTS:
    top = top_next_tokens(model, tok, prompt, k=3)
    p = next_token_prob(tok, answer)(model(**tok(prompt, return_tensors="pt").to(device)))["prob"]
    rows.append({"fakt": name, "P(odp)": p, "top3": list(top["token"]),
                 "podmiot": find_token_positions(tok, prompt, subj)})
pd.DataFrame(rows)

In [ ]:
facts = {}
for name, prompt, subj, answer in FACTS:
    facts[name] = trace_all_streams(
        model, tok, prompt, find_token_positions(tok, prompt, subj), next_token_prob(tok, answer),
        out_dir=OUT, name=name, noise_level=NOISE,
    )

In [ ]:
for name, res in facts.items():
    plot_trace_streams(res, save_path=f"{FIGURES}/{name}.png")

In [ ]:
plot_trace_grid(facts, save_path=f"{FIGURES}/facts_grid.png");

## Analiza

In [ ]:
summary = pd.concat(
    {name: streams_summary(res) for name, res in {"space_needle": space, **facts}.items()},
    names=["fakt", None],
).droplevel(1)
summary

In [ ]:
# które tokeny niosą fakt: maksymalny efekt pośredni na pozycję (strumień MLP)
pd.concat({name: position_summary(res["mlp_activation"]).head(4) for name, res in facts.items()})

## Pytania pomocnicze

**Na czym polega technika Causal Tracing?**

TODO

**Które tokeny podmiotu mają największy wpływ na predykcję?**

TODO (na podstawie `position_summary`: ostatni token podmiotu? pierwszy?)

**Dlaczego wzorce aktywacji dla `attention_output`, `block_output` i `mlp_activation` różnią się?**

TODO (block_output = cały strumień rezydualny, więc przywrócenie niesie wszystko „do tej pory”; MLP = wkład pamięci faktów w środkowych warstwach na podmiocie; atencja = przenoszenie informacji z podmiotu na ostatni token w późnych warstwach)

## Podsumowanie

TODO